# Anti-bot Crawler
Crawl web co bao ve (Playwright Stealth)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK_DIR = '/content/drive/MyDrive/CrawlData'
DOWNLOAD_DIR = os.path.join(WORK_DIR, 'downloaded_files')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print('Work:', WORK_DIR)

In [ ]:
!pip install playwright playwright-stealth beautifulsoup4 lxml aiohttp nest_asyncio -q
!playwright install chromium
!playwright install-deps  # Fix error: missing dependencies for stealth browser

In [ ]:
import asyncio, random, re, json, aiohttp, nest_asyncio
from playwright.async_api import async_playwright
from playwright_stealth import stealth_async
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, unquote
from datetime import datetime
nest_asyncio.apply()

INPUT_FILE = os.path.join(WORK_DIR, 'antibot_urls.json')
OUTPUT_FILE = os.path.join(WORK_DIR, 'output_antibot.json')
USER_AGENTS = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0', 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36']

In [ ]:
def get_filename(url):
    path = urlparse(url).path
    filename = unquote(os.path.basename(path))
    if not filename or filename == '/': filename = f'file_{hash(url) % 100000}'
    if '.' not in filename: filename = filename + '.bin'
    return filename

def is_downloadable(url, exts): return any(urlparse(url.lower()).path.endswith(e) for e in exts)
def is_same_domain(url, base): return urlparse(base).netloc == urlparse(url).netloc

def extract_links(html, base, selector=None, exts=None, require_ext=False):
    soup = BeautifulSoup(html, 'lxml')
    links, seen = [], set()
    if selector and selector.get('selector') and selector.get('type') == 'css':
        for elem in soup.select(selector['selector']):
            if elem.name == 'a' and elem.get('href'):
                u = urljoin(base, elem['href'])
                if u not in seen: seen.add(u); links.append(u)
            for a in elem.find_all('a', href=True):
                u = urljoin(base, a['href'])
                if u not in seen: seen.add(u); links.append(u)
    else:
        for a in soup.find_all('a', href=True):
            u = urljoin(base, a['href'])
            if u not in seen:
                if require_ext and exts:
                    if is_downloadable(u, exts): seen.add(u); links.append(u)
                else: seen.add(u); links.append(u)
    return links

In [ ]:
async def download_cookies(url, folder, fname, cookies):
    r = {'url': url, 'filename': fname, 'status': 'pending'}
    try:
        headers = {'User-Agent': random.choice(USER_AGENTS), 'Cookie': '; '.join([f"{c['name']}={c['value']}" for c in cookies])}
        async with aiohttp.ClientSession() as s:
            async with s.get(url, headers=headers, timeout=aiohttp.ClientTimeout(120)) as resp:
                if resp.status == 200:
                    if 'Content-Disposition' in resp.headers:
                        for p in [r"filename\*=UTF-8''(.+)", r'filename="(.+)"', r"filename='(.+)'", r'filename=([^;\s]+)']:
                            m = re.search(p, resp.headers['Content-Disposition'])
                            if m: r['filename'] = unquote(m.group(1).strip()); break
                    path = os.path.join(folder, r['filename'])
                    c = 1; b, ext = os.path.splitext(path)
                    while os.path.exists(path): path = f'{b}_{c}{ext}'; c += 1
                    content = await resp.read()
                    with open(path, 'wb') as f: f.write(content)
                    r['status'], r['size'], r['path'] = 'success', len(content), path
                else: r['status'], r['error'] = 'failed', f'HTTP {resp.status}'
    except Exception as e: r['status'], r['error'] = 'failed', str(e)
    return r

In [ ]:
async def crawl_multi_level(cfg, options):
    url = cfg.get('url')
    levels = cfg.get('levels', [])
    exts = cfg.get('file_extensions', ['.pdf'])
    max_files = options.get('max_files', 0)
    result = {'url': url, 'levels': [], 'files': [], 'status': 'pending'}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=['--disable-blink-features=AutomationControlled'])
        ctx = await browser.new_context(user_agent=random.choice(USER_AGENTS))
        page = await ctx.new_page()
        await stealth_async(page)
        try:
            current_urls = [url]
            cookies = []
            for level_idx, level_cfg in enumerate(levels):
                level_name = level_cfg.get('name', f'Level {level_idx+1}')
                is_download = level_cfg.get('is_download', False)
                selector = level_cfg.get('selector')
                max_pages = level_cfg.get('max_pages', 30)
                print(f'\n  Level {level_idx+1}: {level_name} ({len(current_urls)} urls)')
                all_links, crawled = [], []
                for i, u in enumerate(current_urls[:max_pages], 1):
                    print(f'  [{i}/{min(len(current_urls), max_pages)}] {u[:50]}...')
                    try:
                        await asyncio.sleep(random.uniform(2, 4))
                        await page.goto(u, wait_until='domcontentloaded', timeout=60000)
                        await page.wait_for_timeout(3000)
                        await page.mouse.wheel(0, random.randint(100, 300))
                        html = await page.content()
                        cookies = await ctx.cookies()
                        links = extract_links(html, u, selector, exts, require_ext=is_download and not selector)
                        if not is_download: links = [l for l in links if is_same_domain(l, url) and l != u]
                        print(f'      -> {len(links)} links')
                        crawled.append({'url': u, 'links_found': len(links), 'status': 'success'})
                        all_links.extend(links)
                    except Exception as e:
                        print(f'      Error: {str(e)[:40]}')
                        crawled.append({'url': u, 'status': 'failed', 'error': str(e)})
                seen = set()
                unique = [l for l in all_links if l not in seen and not seen.add(l)]
                result['levels'].append({'name': level_name, 'pages_crawled': len(crawled), 'links_found': len(unique)})
                print(f'  Total: {len(unique)} links')
                if is_download:
                    if max_files > 0: unique = unique[:max_files]; print(f'  Limited to {max_files} files')
                    print(f'\n  Downloading {len(unique)} files...')
                    for i, dl_url in enumerate(unique):
                        await asyncio.sleep(random.uniform(1, 3))
                        dl = await download_cookies(dl_url, DOWNLOAD_DIR, get_filename(dl_url), cookies)
                        result['files'].append(dl)
                        status = 'OK' if dl['status']=='success' else 'FAIL'
                        print(f'    [{i+1}] [{status}] {dl["filename"][:40]}')
                    break
                else:
                    current_urls = unique
                    if not current_urls: print('  No more URLs'); break
            result['status'] = 'success'
        except Exception as e: result['status'], result['error'] = 'failed', str(e)
        finally: await browser.close()
    return result

async def crawl_two_level(cfg, options):
    cfg['levels'] = [{'name': 'Detail pages', 'selector': cfg.get('level1_selector'), 'max_pages': cfg.get('max_detail_pages', 30)}, {'name': 'Download links', 'selector': cfg.get('level2_selector'), 'is_download': True}]
    return await crawl_multi_level(cfg, options)

async def crawl_one_level(cfg, options):
    cfg['levels'] = [{'name': 'Download', 'selector': cfg.get('region_selector') or cfg.get('level2_selector'), 'is_download': True}]
    return await crawl_multi_level(cfg, options)

In [ ]:
async def main():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f: data = json.load(f)
    urls, opts = data.get('urls', []), data.get('options', {})
    print(f'URLs: {len(urls)}')
    if opts.get('max_files'): print(f'Max files: {opts["max_files"]}')
    print('='*50)
    results = []
    for i, cfg in enumerate(urls, 1):
        url, mode = cfg.get('url'), cfg.get('crawl_mode', 'one_level')
        print(f'\n[{i}] {url[:50]}...')
        print(f'  Mode: {mode}')
        if mode == 'multi_level': r = await crawl_multi_level(cfg, opts)
        elif mode == 'two_level': r = await crawl_two_level(cfg, opts)
        else: r = await crawl_one_level(cfg, opts)
        ok = sum(1 for f in r['files'] if f['status']=='success')
        print(f'\n  Downloaded: {ok}/{len(r["files"])}')
        results.append(r)
        if i < len(urls):
            delay = random.randint(5, 10)
            print(f'  Wait {delay}s...')
            await asyncio.sleep(delay)
    return results

all_results = asyncio.get_event_loop().run_until_complete(main())
print('\nDone!')

In [ ]:
output = {'results': all_results, 'summary': {'urls': len(all_results), 'total_files': sum(len(r['files']) for r in all_results), 'downloaded': sum(sum(1 for f in r['files'] if f['status']=='success') for r in all_results)}, 'crawled_at': datetime.now().isoformat()}
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f: json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Total: {output["summary"]["total_files"]}')
print(f'Downloaded: {output["summary"]["downloaded"]}')